# 01. MobileFaceNet FR 베이스라인 학습 (ArcFace, AdaFace, MagFace)

본 노트북은 UCFace 논문(IEEE T-IFS 2024, DOI: [10.1109/TIFS.2024.3426973](https://doi.org/10.1109/TIFS.2024.3426973))에서 보고된
동일 백본(MobileFaceNet 512D) 기반의 3대 대표 얼굴인식 마진 손실함수(ArcFace, AdaFace, MagFace)를 재현 학습합니다.

### 손실 함수 공식 및 하이퍼파라미터 규격
1. **ArcFace** (Deng et al., CVPR 2019, DOI: [10.1109/CVPR.2019.00482](https://doi.org/10.1109/CVPR.2019.00482))
   - 마진 $m = 0.6$, 스케일 $s = 64.0$
   - $\mathcal{L}_{\text{arc}} = -\log \frac{e^{s \cos(\theta_{y_i} + m)}}{e^{s \cos(\theta_{y_i} + m)} + \sum_{j \neq y_i} e^{s \cos \theta_j}}$
2. **AdaFace** (Kim et al., CVPR 2022, DOI: [10.1109/CVPR.2022.01011](https://doi.org/10.1109/CVPR.2022.01011))
   - 마진 $m = 0.4$, 스케일 $s = 64.0$, 스케일링 계수 $h = 0.333$
   - 동적 마진: $g_{\text{angle}} = -m \cdot \widehat{\|z_i\|}, \quad g_{\text{add}} = m \cdot \widehat{\|z_i\|} + m$
   - 특징 벡터 노름(Norm) EMA 평균 $\mu_z$, 표준편차 $\sigma_z$ 추적을 통한 화질 적응형 마진 부여
3. **MagFace** (Meng et al., CVPR 2021, DOI: [10.1109/CVPR.2021.01170](https://doi.org/10.1109/CVPR.2021.01170))
   - $l_a = 10.0, u_a = 110.0, l_m = 0.45, u_m = 0.8, \lambda_g = 20.0, s = 64.0$
   - 적응형 마진 $m(a_i) = \frac{u_m - l_m}{u_a - l_a}(a_i - l_a) + l_m$, 크기 정규화 $\mathcal{L}_{g} = \frac{1}{a_i} + \frac{a_i}{u_a^2}$

### 학습 레시피
- 옵티마이저: Adam (초기 lr $10^{-3}$, weight decay $5 \times 10^{-4}$)
- 스케줄러: Epoch 10, 20, 30에서 $0.1\times$ 감쇄 (총 40 epochs)
- 배치 크기: 64 (GTX 1080 Ti 11GB VRAM 안전을 위해 Micro-batch 32 + Gradient Accumulation 2회 적용)
- 백본 드롭아웃: 0.6 (UCFace 실험 프로토콜 준수)

In [ ]:
# cell 1 : 기본 라이브러리 임포트 및 하드웨어 점검
from __future__ import annotations
from pathlib import Path
import sys
import torch
import pandas as pd

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.training.fr_baselines import (
    FRBaselineTrainer,
    FRBaselineTrainingModule,
    FRDataset,
    build_fr_baseline_model,
    export_backbone_checkpoint,
)
from research.embeddings.pytorch.official_loaders import load_mobilefacenet_checkpoint

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    device = "cuda"
else:
    print("Warning: GPU not detected. Using CPU.")
    device = "cpu"

In [ ]:
# cell 2 : 실험 설정 및 하이퍼파라미터 정의 (표준 설정 인터페이스 계약)
# -------------------------------------------------------------------------
# 1. 데이터 및 출력 루트 경로
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "results/training/fr_baselines"

# 2. 학습 단계, 기법 및 시드
STAGE = "finetune"                           # 'finetune' (SurvFace) 또는 'pretrain' (VGGFace2)
METHODS = ["arcface", "adaface", "magface"]   # 순차 학습 및 비교 대상 FR 손실함수 헤드
SEEDS = [8972]                               # 재현성 보장 시드

# 3. 연산 장치 및 VRAM 최적화 마이크로배치 설정 (GTX 1080 Ti 11GB 기준)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MICRO_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2                        # 실효 배치 크기 = 32 * 2 = 64 (UCFace 논문 규격)

# 4. 실행 가드 (안전 기본값: False - 의도치 않은 대규모 연산 자동 시작 방지)
EXECUTE_TRAINING = False                     # 사용자의 명시적 승인 후 True로 전환
EXECUTE_FULL_EVALUATION = False              # 학습 완료 후 다운스트림 전체 평가 자동 실행 방지
DRY_RUN_SMOKE = True                         # 빠른 파이프라인 무결성 검증용 스모크 모드 (1 epoch, 2 batch)

# 5. 학습 재개 및 원본 결과 보존 정책
RESUME_RUN = None                            # 재개 시 체크포인트 경로 지정 (e.g. 'checkpoint_latest.pt')
KEEP_RAW_RESULTS = False                     # False: 최신/최고/최종만 보존하여 디스크 용량 절약 (AGENTS.md 준수)

# 6. 학습 하이퍼파라미터 (UCFace 및 논문 명시 규격)
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 5e-4
DECAY_EPOCHS = (10, 20, 30)
DROPOUT = 0.6                                # UCFace MobileFaceNet 규격 (conv_sep -> features)
COLOR_ORDER = "bgr"                          # FaceX-Zoo 및 저장소 inference 계약과 100% 일치

MANIFEST_PATH = DATA_ROOT / "manifests/training_manifest_v2_finetune.csv"
print(f"STAGE={STAGE}, METHODS={METHODS}, SEEDS={SEEDS}")
print(f"DEVICE={DEVICE}, MICRO_BATCH_SIZE={MICRO_BATCH_SIZE}, GRAD_ACCUM_STEPS={GRAD_ACCUM_STEPS}")
print(f"EXECUTE_TRAINING={EXECUTE_TRAINING}, DRY_RUN_SMOKE={DRY_RUN_SMOKE}, KEEP_RAW_RESULTS={KEEP_RAW_RESULTS}")


In [ ]:
# cell 3 : 데이터셋 및 데이터로더 초기화
if DRY_RUN_SMOKE or not MANIFEST_PATH.is_file():
    print("[Mode: Synthetic Smoke] Initializing synthetic dataset for pipeline validation.")
    train_ds = FRDataset(synthetic=True, synthetic_samples=64, synthetic_classes=8, color_order=COLOR_ORDER, is_train=True)
    val_ds = FRDataset(synthetic=True, synthetic_samples=32, synthetic_classes=8, color_order=COLOR_ORDER, is_train=False)
    num_classes = train_ds.num_classes
else:
    print(f"[Mode: Real Data] Loading dataset from {MANIFEST_PATH} (STAGE={STAGE})")
    manifest_df = pd.read_csv(MANIFEST_PATH)
    train_ds = FRDataset(manifest_df, project_root=PROJECT_ROOT, finetune_role="train", color_order=COLOR_ORDER, is_train=True)
    val_ds = FRDataset(manifest_df, project_root=PROJECT_ROOT, finetune_role="val", color_order=COLOR_ORDER, is_train=False)
    num_classes = train_ds.num_classes

print(f"Classes: {num_classes}, Train samples: {len(train_ds)}, Val samples: {len(val_ds)}, Color order: {COLOR_ORDER}")


In [ ]:
# cell 4 : 공통 사전학습 MobileFaceNet 백본 초기화 (동일 시드 다중 헤드 분기용)
from research.embeddings.pytorch.official_backbones import build_mobilefacenet_backbone
common_seed = SEEDS[0] if SEEDS else 8972
torch.manual_seed(common_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(common_seed)

# 동일 사전학습 가중치를 복제하기 위한 공통 기본 백본 인스턴스 준비
base_backbone = build_mobilefacenet_backbone("mobilefacenet", embedding_dim=512, dropout=DROPOUT)
backbone_params = sum(p.numel() for p in base_backbone.parameters())
print(f"Base MobileFaceNet backbone initialized with seed={common_seed}")
print(f"MobileFaceNet backbone parameter count: {backbone_params:,} (~1.2M)")


In [ ]:
# cell 5 : 3대 베이스라인(ArcFace, AdaFace, MagFace) 순차 학습 실행기
training_results = {}

for method in METHODS:
    for seed in SEEDS:
        run_name = f"{method}_seed_{seed}"
        out_dir = OUTPUT_ROOT / method
        print(f"\n>>> [Pipeline] Preparing run: {run_name} (out_dir={out_dir})")

        # 동일 시드 하에서 공통 백본 가중치를 정확히 복제하여 분기
        torch.manual_seed(seed)
        model = build_fr_baseline_model(
            head_type=method,
            num_classes=num_classes,
            architecture="mobilefacenet",
            embedding_dim=512,
            dropout=DROPOUT,
            pretrained_backbone=base_backbone,
        )

        trainer = FRBaselineTrainer(
            model=model,
            train_dataset=train_ds,
            val_dataset=val_ds,
            micro_batch_size=MICRO_BATCH_SIZE,
            grad_accum_steps=GRAD_ACCUM_STEPS,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            decay_epochs=DECAY_EPOCHS,
            epochs=EPOCHS,
            device=DEVICE,
            output_dir=out_dir,
            keep_raw_results=KEEP_RAW_RESULTS,
            resume_checkpoint=RESUME_RUN,
            dry_run=DRY_RUN_SMOKE,
        )

        if EXECUTE_TRAINING or DRY_RUN_SMOKE:
            print(f"[Training Started] Training {method} (dry_run={DRY_RUN_SMOKE})...")
            summary = trainer.fit()
            training_results[run_name] = summary
            print(f"[Training Finished] {run_name}: final_backbone={summary['final_backbone']}")
        else:
            print(f"[Training Skipped] EXECUTE_TRAINING=False. Set EXECUTE_TRAINING=True to train {method}.")


In [ ]:
# cell 6 : 학습 지표 및 수렴 이력 확인
import json
for method in METHODS:
    metrics_file = OUTPUT_ROOT / method / "metrics_history.json"
    if metrics_file.is_file():
        with open(metrics_file, "r", encoding="utf-8") as f:
            h_data = json.load(f)
        df_h = pd.DataFrame(h_data.get("history", []))
        print(f"\n[{method.upper()} Metrics History Summary]")
        print(df_h.head(5))
    else:
        print(f"[{method.upper()}] Metrics history file not found yet.")


In [ ]:
# cell 7 : 최종 백본 체크포인트 내보내기 및 공식 로더 검증
from research.embeddings.base import CheckpointProvenance, ModelSpec, PreprocessingSpec
for method in METHODS:
    export_file = OUTPUT_ROOT / method / "backbone_final.pt"
    if export_file.is_file():
        test_spec = ModelSpec(
            family=method,
            architecture="mobilefacenet",
            training_dataset="training_export",
            implementation_repository="research.training.fr_baselines",
            checkpoint=CheckpointProvenance.from_file(
                export_file,
                source_url="local://trained",
            ),
            preprocessing=PreprocessingSpec(
                input_height=112,
                input_width=112,
                source_color_order="rgb",
                model_color_order=COLOR_ORDER,
                channel_mean=(127.5, 127.5, 127.5),
                channel_std=(128.0, 128.0, 128.0),
            ),
            target_layer="conv_sep.conv",
            embedding_dim=512,
        )
        loaded_bb = load_mobilefacenet_checkpoint(test_spec)
        print(f"[Validation Pass] {method}: {export_file} verified with {type(loaded_bb).__name__}")
    else:
        print(f"[{method}] Export file {export_file} not yet generated.")
